In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1] # go up n levels (adjust as needed)
sys.path.append(str(ROOT))

from config import PROJECT_ROOT, APT_ROOT
from apt_project import *

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
import pandas as pd

In [3]:
# Define feature columns
feature_cols = ['year', 'month', 'industry_code', 'event_type', 'event_subtype', 'motive', 'actor_type']

# Identify Categorical vs Numeric
categorical_cols = ['industry_code', 'event_type', 'event_subtype', 'motive', 'actor_type']
numeric_cols =['year', 'month']

In [4]:
# Preprocess with one-hot encoding (fit on all data)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

# Fit preprocessor on full dataset
preprocessor.fit(events_df[feature_cols])

,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


In [5]:
# Filter out NaN values for the respective training sets
train_df_origin = events_df[events_df['origin_risk_norm'].notna()]
train_df_victim = events_df[events_df['victim_risk_norm'].notna()]

In [6]:
# Target Variables (y)
y_origin = train_df_origin['origin_risk_norm']
y_victim = train_df_victim['victim_risk_norm']

# Input Features (X)
X_origin_raw = train_df_origin[feature_cols]
X_origin = preprocessor.transform(X_origin_raw)
X_victim_raw = train_df_victim[feature_cols]
X_victim = preprocessor.transform(X_victim_raw)

In [7]:
# Train models
origin_model = GradientBoostingRegressor()
victim_model = GradientBoostingRegressor()

origin_model.fit(X_origin, y_origin)
victim_model.fit(X_victim, y_victim)

,loss,'squared_error'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [8]:
# Predict for all events
X_all = preprocessor.transform(events_df[feature_cols])

events_df['ml_origin_risk'] = origin_model.predict(X_all)
events_df['ml_target_risk'] = victim_model.predict(X_all)

In [9]:
#events_df

## Extrapolating to the Other APTs

In [10]:
from sklearn.preprocessing import MultiLabelBinarizer

# Group technique names by APT group_name
grouped = (
    group_techniques_df
    .groupby("group_name")["technique_name"]
    .apply(list)
)

# Multi-label binarize
mlb = MultiLabelBinarizer()
tech_matrix = mlb.fit_transform(grouped)

# Build the technique matrix DataFrame
group_tech_matrix_df = pd.DataFrame(
    tech_matrix,
    index=grouped.index,      # IMPORTANT: use .index, not grouped directly
    columns=mlb.classes_
)

#group_tech_matrix_df.tail(10)

In [11]:
print(events_df['apt_group'].dropna().unique()[:50])
print(group_techniques_df['group_name'].unique()[:50])

['dragonok' 'cleaver' 'apt3' 'apt32' 'apt19' 'apt29' 'charming kitten'
 'threat group-3390' 'dragonfly 2.0' 'carbanak' 'menupass' 'ke3chang'
 'leviathan' 'fin7' 'bronze butler' 'darkhydrus' 'turla' 'apt33'
 'zirconium' 'windshift' 'apt39' 'ta505' 'fin8' 'silence' 'machete'
 'apt17' 'lapsus$' 'rancor' 'apt30' 'putter panda' 'mustang panda'
 'agrius' 'sidewinder' 'mustard tempest' 'ember bear' 'saint bear'
 'blackbyte' 'sandworm team' 'bitter' 'confucius' 'magic hound'
 'silverterrier' 'transparent tribe' 'temp.veles' 'winter vivern'
 'sidecopy' 'applejeus' 'muddywater' 'akira' 'scattered spider']
['Indrik Spider' 'LuminousMoth' 'Medusa Group' 'Wizard Spider' 'FIN7'
 'UNC3886' 'WIRTE' 'Dragonfly' 'Equation' 'OilRig' 'Fox Kitten'
 'Aquatic Panda' 'Daggerfly' 'TeamTNT' 'Inception' 'BlackTech' 'APT42'
 'Malteiro' 'Earth Lusca' 'Play' 'Sandworm Team' 'Turla' 'Ember Bear'
 'Silence' 'Patchwork' 'APT28' 'Cinnamon Tempest' 'HEXANE' 'Darkhotel'
 'Ke3chang' 'Volt Typhoon' 'Magic Hound' 'Cobalt Gr

In [13]:
alias_to_official

{'indrik spider': 'Indrik Spider',
 'evil corp': 'Indrik Spider',
 'manatee tempest': 'Indrik Spider',
 'dev-0243': 'Indrik Spider',
 'unc2165': 'Indrik Spider',
 'luminousmoth': 'LuminousMoth',
 'medusa group': 'Medusa Group',
 'wizard spider': 'Wizard Spider',
 'unc1878': 'Wizard Spider',
 'temp.mixmaster': 'Wizard Spider',
 'grim spider': 'Wizard Spider',
 'fin12': 'Wizard Spider',
 'gold blackburn': 'Wizard Spider',
 'itg23': 'Wizard Spider',
 'periwinkle tempest': 'Wizard Spider',
 'dev-0193': 'Wizard Spider',
 'elderwood': 'Elderwood',
 'elderwood gang': 'Elderwood',
 'beijing group': 'Elderwood',
 'sneaky panda': 'Elderwood',
 'frankenstein': 'Frankenstein',
 'fin7': 'FIN7',
 'gold niagara': 'FIN7',
 'itg14': 'FIN7',
 'carbon spider': 'FIN7',
 'elbrus': 'FIN7',
 'sangria tempest': 'FIN7',
 'unc3886': 'UNC3886',
 'velvet ant': 'Velvet Ant',
 'wirte': 'WIRTE',
 'dragonfly': 'Dragonfly',
 'temp.isotope': 'Dragonfly',
 'dymalloy': 'Dragonfly 2.0',
 'berserk bear': 'Dragonfly 2.0',
 